# Naive Bayes for Conspiracy Detection
## Predicting conspiracy labels and marker types

This notebook implements Naive Bayes classifier for:
1. **Conspiracy Label Prediction**: yes/no/cant_tell (multiclass)
2. **Marker Type Prediction**: Action, Actor, Effect, Evidence, Victim (binary classification for each)


In [10]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")


Libraries imported successfully!


In [23]:
# Load and merge all features
BASE = Path('../')
PROC = BASE / 'data_processed'
FEAT = BASE / 'features'

# Load base data
#df = pd.read_parquet(PROC / 'data_processed.parquet')
#if df is None or len(df) == 0:

df = pd.read_csv(PROC / 'data_clean.csv')

print(f"Base data: {df.shape}")

# Load feature files
feature_files = [
    FEAT / 'lexical_complexity.csv',
    FEAT / 'discourse_markers.csv',
    FEAT / 'sentiment_emotion.csv',
    FEAT / 'token_pos_counts.csv',
    FEAT / 'readability_scores.csv',
    FEAT / 'ma_ttr_mtld_scores.csv',
    FEAT / 'pos_analysis.csv'
]

merged = df.copy()
for feat_file in feature_files:
    if feat_file.exists():
        try:
            feat_df = pd.read_csv(feat_file)
            if '_id' in merged.columns and '_id' in feat_df.columns:
                merged = merged.merge(feat_df, on='_id', how='left')
                print(f"Merged {feat_file.name}")
        except Exception as e:
            print(f"Could not load {feat_file.name}: {e}")

print(f"\nFinal merged shape: {merged.shape}")
print(f"Columns: {len(merged.columns)}")
print(merged.head())


Base data: (4316, 5)
Merged lexical_complexity.csv
Merged discourse_markers.csv
Merged sentiment_emotion.csv
Merged readability_scores.csv
Merged ma_ttr_mtld_scores.csv
Merged pos_analysis.csv

Final merged shape: (701768, 68)
Columns: 68
          _id                                               text  \
0  t1_f7ju17o  A great article on what's taking place in Boli...   
1  t1_k5c5yyz  Chris Lehto interviews Ashton Forbes about his...   
2  t1_givys64  Germany has upset other EU member states by se...   
3  t1_joq538t  Redditors are, just like most social media use...   
4  t1_hlk1vci  u/DLWzll shared a couple days ago how the Virg...   

         subreddit conspiracy  \
0       conspiracy        yes   
1  HighStrangeness  cant_tell   
2           europe         no   
3       Republican  cant_tell   
4  ForwardPartyUSA  cant_tell   

                                             markers       ttr  hapax_ratio  \
0  [{'startIndex': 8, 'endIndex': 15, 'type': 'Ev...  0.857143     0.71428

In [13]:
# Extract numeric features
numeric_cols = merged.select_dtypes(include=[np.number]).columns.tolist()
# Remove _id and other non-feature columns
numeric_cols = [c for c in numeric_cols if c not in ['_id']]
print(f"Number of numeric features: {len(numeric_cols)}")

X = merged[numeric_cols].fillna(0)
print(f"Feature matrix shape: {X.shape}")
print(f"Missing values: {X.isna().sum().sum()}")


Number of numeric features: 72
Feature matrix shape: (185682, 72)
Missing values: 0


## Task 1: Conspiracy Label Prediction (yes/no/cant_tell)


In [14]:
# Task 1: Conspiracy Label Prediction (yes/no/cant_tell)
y_conspiracy = merged['conspiracy'].copy()
# Filter out any null labels
mask = y_conspiracy.notna()
X_clean = X[mask]
y_clean = y_conspiracy[mask]

print(f"Samples for conspiracy prediction: {len(y_clean)}")
print(f"Label distribution:")
print(y_clean.value_counts())


Samples for conspiracy prediction: 185682
Label distribution:
conspiracy
yes          83413
cant_tell    54529
no           47740
Name: count, dtype: int64


In [15]:
# Encode labels
le_conspiracy = LabelEncoder()
y_encoded = le_conspiracy.fit_transform(y_clean)
print(f"Encoded labels: {dict(zip(le_conspiracy.classes_, range(len(le_conspiracy.classes_))))}")

# Create pipeline with scaling and Naive Bayes
# Use GaussianNB for continuous features
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('nb', GaussianNB())
])

# 5-fold cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = ['accuracy', 'f1_macro', 'f1_weighted', 'precision_macro', 'recall_macro']
cv_results = cross_validate(pipeline, X_clean, y_encoded, cv=cv, scoring=scoring, n_jobs=-1)

print("\n=== Conspiracy Label Prediction (5-fold CV) ===")
for metric in scoring:
    scores = cv_results[f'test_{metric}']
    print(f"{metric}: {scores.mean():.4f} (+/- {scores.std() * 2:.4f})")

# Train final model
pipeline.fit(X_clean, y_encoded)
print("\nModel trained successfully!")


Encoded labels: {'cant_tell': 0, 'no': 1, 'yes': 2}

=== Conspiracy Label Prediction (5-fold CV) ===
accuracy: 0.7345 (+/- 0.0827)
f1_macro: 0.6996 (+/- 0.0915)
f1_weighted: 0.7192 (+/- 0.0884)
precision_macro: 0.7607 (+/- 0.0390)
recall_macro: 0.6950 (+/- 0.0981)

Model trained successfully!


## Task 2: Marker Span Extraction (only for conspiracy="yes")

For documents with conspiracy="yes", extract marker spans from text:
- Action, Actor, Effect, Evidence, Victim
- Each marker has: startIndex, endIndex, type, text


In [17]:
# Task 2: Marker Span Extraction (only for conspiracy="yes")
MARKER_TYPES = ["Action", "Actor", "Effect", "Evidence", "Victim"]

# Filter to only conspiracy="yes" samples for marker extraction
yes_mask = y_clean == 'yes'
X_yes = X_clean[yes_mask]
merged_yes = merged[mask][yes_mask].reset_index(drop=True)

print(f"Conspiracy='yes' samples: {len(X_yes)}")
print(f"\nProcessing marker span extraction for 'yes' samples only...")

# Extract marker data for yes samples
def extract_marker_info(row):
    """Extract marker information from row"""
    markers = row.get('markers', [])
    if pd.isna(markers) or markers == '':
        return []
    
    if isinstance(markers, str):
        try:
            markers = json.loads(markers)
        except:
            return []
    
    if isinstance(markers, list):
        return [m for m in markers if isinstance(m, dict) and 'type' in m and m['type'] in MARKER_TYPES]
    
    return []

marker_info_yes = merged_yes.apply(extract_marker_info, axis=1)

# Count markers by type
marker_counts_by_type = {mt: 0 for mt in MARKER_TYPES}
total_markers = 0
for markers in marker_info_yes:
    total_markers += len(markers)
    for m in markers:
        if m['type'] in marker_counts_by_type:
            marker_counts_by_type[m['type']] += 1

print(f"\nTotal markers found: {total_markers}")
print("Markers by type:")
for mt in MARKER_TYPES:
    print(f"  {mt}: {marker_counts_by_type[mt]}")


Conspiracy='yes' samples: 83413

Processing marker span extraction for 'yes' samples only...

Total markers found: 0
Markers by type:
  Action: 0
  Actor: 0
  Effect: 0
  Evidence: 0
  Victim: 0


In [18]:
# Train binary classifiers to predict marker presence for each type
# Then use text features to help extract spans

marker_models = {}

for marker_type in MARKER_TYPES:
    # Create binary labels: 1 if document has this marker type, 0 otherwise
    y_marker_presence = marker_info_yes.apply(
        lambda markers: 1 if any(m['type'] == marker_type for m in markers) else 0
    ).values
    
    print(f"\n{marker_type} presence: {y_marker_presence.sum()} / {len(y_marker_presence)} documents")
    
    if y_marker_presence.sum() < 2:  # Need at least 2 positive samples
        print(f"  Skipping {marker_type} - insufficient positive samples")
        continue
    
    # Train model to predict marker presence
    pipeline_marker = Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler()),
        ('nb', GaussianNB())
    ])
    
    cv_marker = StratifiedKFold(n_splits=min(5, y_marker_presence.sum()), shuffle=True, random_state=42)
    
    try:
        cv_scores = cross_validate(pipeline_marker, X_yes, y_marker_presence, 
                                 cv=cv_marker, scoring=['accuracy', 'f1', 'precision', 'recall'], 
                                 n_jobs=-1)
        
        print(f"  Accuracy: {cv_scores['test_accuracy'].mean():.4f} (+/- {cv_scores['test_accuracy'].std() * 2:.4f})")
        print(f"  F1: {cv_scores['test_f1'].mean():.4f} (+/- {cv_scores['test_f1'].std() * 2:.4f})")
        print(f"  Precision: {cv_scores['test_precision'].mean():.4f} (+/- {cv_scores['test_precision'].std() * 2:.4f})")
        print(f"  Recall: {cv_scores['test_recall'].mean():.4f} (+/- {cv_scores['test_recall'].std() * 2:.4f})")
        
        # Train final model
        pipeline_marker.fit(X_yes, y_marker_presence)
        marker_models[marker_type] = pipeline_marker
        
    except Exception as e:
        print(f"  Error training {marker_type}: {e}")

print(f"\n=== Marker Presence Models Trained: {len(marker_models)}/{len(MARKER_TYPES)} ===")

# Helper function to extract marker spans from text
# This is a simplified approach - in practice, you might use NER or more sophisticated methods
def extract_marker_spans_simple(text, marker_type, model, features_dict):
    """
    Simplified marker span extraction.
    In practice, this should use more sophisticated methods like NER, POS tagging, etc.
    """
    # For now, return empty list - this would need to be implemented with actual text processing
    # The model above predicts presence, but actual span extraction requires text analysis
    return []

print("\nNote: Marker span extraction (startIndex, endIndex) requires text analysis.")
print("The models above predict marker presence. For actual span extraction,")
print("you would need to combine these predictions with text processing (NER, POS, etc.)")



Action presence: 0 / 83413 documents
  Skipping Action - insufficient positive samples

Actor presence: 0 / 83413 documents
  Skipping Actor - insufficient positive samples

Effect presence: 0 / 83413 documents
  Skipping Effect - insufficient positive samples

Evidence presence: 0 / 83413 documents
  Skipping Evidence - insufficient positive samples

Victim presence: 0 / 83413 documents
  Skipping Victim - insufficient positive samples

=== Marker Presence Models Trained: 0/5 ===

Note: Marker span extraction (startIndex, endIndex) requires text analysis.
The models above predict marker presence. For actual span extraction,
you would need to combine these predictions with text processing (NER, POS, etc.)


In [19]:
# Implement actual span extraction using text processing
# This combines ML predictions with text analysis to extract marker spans

try:
    import spacy
    try:
        nlp = spacy.load("en_core_web_sm")
        print("spaCy model loaded successfully")
    except OSError:
        print("spaCy model not found. Install with: python -m spacy download en_core_web_sm")
        nlp = None
except ImportError:
    print("spaCy not installed. Install with: pip install spacy")
    nlp = None

def extract_marker_spans_from_text(text, marker_type, has_marker):
    """
    Extract marker spans from text for a given marker type.
    
    Args:
        text: Input text string
        marker_type: Type of marker to extract (Action, Actor, Effect, Evidence, Victim)
        has_marker: Boolean from ML model prediction
        
    Returns:
        List of marker dicts with startIndex, endIndex, type, text
    """
    if not has_marker or not text or not nlp:
        return []
    
    doc = nlp(text)
    markers = []
    
    # Simple heuristic approach based on marker type
    # In practice, this should be trained on actual spans or use sequence labeling
    
    if marker_type == "Actor":
        # Extract named entities and pronouns
        for ent in doc.ents:
            if ent.label_ in ["PERSON", "ORG", "GPE", "NORP"]:
                markers.append({
                    "startIndex": ent.start_char,
                    "endIndex": ent.end_char,
                    "type": marker_type,
                    "text": ent.text
                })
    
    elif marker_type == "Action":
        # Extract verbs and verb phrases
        for token in doc:
            if token.pos_ == "VERB" and token.dep_ in ["ROOT", "ccomp", "xcomp"]:
                # Get verb phrase
                start_char = token.idx
                end_char = token.idx + len(token.text)
                markers.append({
                    "startIndex": start_char,
                    "endIndex": end_char,
                    "type": marker_type,
                    "text": token.text
                })
    
    elif marker_type == "Evidence":
        # Extract noun phrases that might be evidence
        for chunk in doc.noun_chunks:
            if any(tok.pos_ == "NOUN" for tok in chunk):
                markers.append({
                    "startIndex": chunk.start_char,
                    "endIndex": chunk.end_char,
                    "type": marker_type,
                    "text": chunk.text
                })
    
    elif marker_type == "Effect":
        # Extract clauses/phrases indicating effects
        for token in doc:
            if token.pos_ == "VERB" and token.dep_ == "ROOT":
                # Look for dependent clauses
                for child in token.children:
                    if child.dep_ in ["nsubjpass", "dobj"]:
                        markers.append({
                            "startIndex": child.idx,
                            "endIndex": child.idx + len(child.text),
                            "type": marker_type,
                            "text": child.text
                        })
    
    elif marker_type == "Victim":
        # Similar to Actor but focus on objects/nsubjpass
        for token in doc:
            if token.dep_ in ["nsubjpass", "dobj", "pobj"]:
                if token.pos_ in ["NOUN", "PROPN"]:
                    markers.append({
                        "startIndex": token.idx,
                        "endIndex": token.idx + len(token.text),
                        "type": marker_type,
                        "text": token.text
                    })
    
    return markers

# Test span extraction on a sample
if len(merged_yes) > 0 and nlp:
    sample_text = merged_yes.iloc[0].get('text', '')
    if sample_text:
        print(f"\nTesting span extraction on sample text (first 200 chars):")
        print(sample_text[:200])
        
        # Test each marker type
        for mt in MARKER_TYPES:
            if mt in marker_models:
                # Predict presence
                sample_features = X_yes.iloc[0:1]
                has_marker = marker_models[mt].predict(sample_features)[0]
                
                if has_marker:
                    spans = extract_marker_spans_from_text(sample_text, mt, has_marker)
                    print(f"\n{mt} (predicted: has marker):")
                    for span in spans[:3]:  # Show first 3
                        print(f"  {span}")
else:
    print("\nSkipping span extraction test - spaCy not available or no samples")


spaCy model loaded successfully

Testing span extraction on sample text (first 200 chars):
A great article on what's taking place in Bolivia, referencing some similar US backed coups in the region as well as recounting some of Bolivia's history and western policy towards the country.


## Testing on Dev Set


In [21]:
# Load dev set for testing
dev_file = Path('../../dev_rehydrated.jsonl')
dev_data = []
with open(dev_file, 'r', encoding='utf-8') as f:
    for line in f:
        dev_data.append(json.loads(line.strip()))

dev_df = pd.DataFrame(dev_data)
print(f"Dev set size: {len(dev_df)}")
print(f"Dev set columns: {dev_df.columns.tolist()}")

print("\n=== Two-Stage Prediction Pipeline ===")
print("1. Predict conspiracy label (yes/no/cant_tell)")
print("2. For predictions='yes', extract marker spans from text")
print("\nNote: Dev set features need to be computed using the EDA pipeline first.")
print("\nExample prediction flow:")
print("  # Stage 1: Conspiracy classification")
print("  conspiracy_pred = le_conspiracy.inverse_transform(pipeline.predict(X_dev))")
print("  ")
print("  # Stage 2: Marker extraction (only for 'yes')")
print("  for idx, pred in enumerate(conspiracy_pred):")
print("      if pred == 'yes':")
print("          text = dev_df.iloc[idx]['text']")
print("          for marker_type in MARKER_TYPES:")
print("              if marker_type in marker_models:")
print("                  has_marker = marker_models[marker_type].predict(X_dev[idx:idx+1])[0]")
print("                  spans = extract_marker_spans_from_text(text, marker_type, has_marker)")


Dev set size: 100
Dev set columns: ['_id', 'text', 'subreddit', 'conspiracy', 'markers', 'annotator']

=== Two-Stage Prediction Pipeline ===
1. Predict conspiracy label (yes/no/cant_tell)
2. For predictions='yes', extract marker spans from text

Note: Dev set features need to be computed using the EDA pipeline first.

Example prediction flow:
  # Stage 1: Conspiracy classification
  conspiracy_pred = le_conspiracy.inverse_transform(pipeline.predict(X_dev))
  
  # Stage 2: Marker extraction (only for 'yes')
  for idx, pred in enumerate(conspiracy_pred):
      if pred == 'yes':
          text = dev_df.iloc[idx]['text']
          for marker_type in MARKER_TYPES:
              if marker_type in marker_models:
                  has_marker = marker_models[marker_type].predict(X_dev[idx:idx+1])[0]
                  spans = extract_marker_spans_from_text(text, marker_type, has_marker)
